In [1]:
!which python

/home/mel/.pyenv/shims/python


In [1]:
# Install
!pip install -q imbalanced-learn openpyxl

# Imports
import os
import glob
import zipfile
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

In [2]:
# Unzip Everything
!unzip -o heartbeat.zip -d heartbeat
!unzip -o patients_data_with_alerts.xlsx.zip

for z in glob.glob("heartbeat/*.csv.zip"):
    with zipfile.ZipFile(z, "r") as zip_ref:
        zip_ref.extractall("heartbeat/")

print("heartbeat folder:", os.listdir("heartbeat"))
print("content root:", os.listdir())

Archive:  heartbeat.zip
  inflating: heartbeat/ptbdb_normal.csv.zip  
  inflating: heartbeat/mitbih_test.csv.zip  
  inflating: heartbeat/mitbih_train.csv.zip  
  inflating: heartbeat/ptbdb_abnormal.csv.zip  
Archive:  patients_data_with_alerts.xlsx.zip
  inflating: patients_data_with_alerts.xlsx  
heartbeat folder: ['mitbih_train.csv.zip', 'ptbdb_abnormal.csv', 'mitbih_train.csv', 'ptbdb_normal.csv', 'mitbih_test.csv.zip', 'ptbdb_normal.csv.zip', 'ptbdb_abnormal.csv.zip', 'mitbih_test.csv']
content root: ['.config', 'patients_data_with_alerts.xlsx.zip', 'heartbeat.zip', 'patients_data_with_alerts.xlsx', 'heartbeat', 'sample_data']


# ECG Pipeline

In [3]:
# Load (MIT-BIH)
train = pd.read_csv("heartbeat/mitbih_train.csv", header=None)
test = pd.read_csv("heartbeat/mitbih_test.csv", header=None)

X_train = train.iloc[:, :-1].values
y_train = train.iloc[:, -1].values.astype(int)
X_test = test.iloc[:, :-1].values
y_test = test.iloc[:, -1].values.astype(int)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (87554, 187) Test: (21892, 187)


In [4]:
# Inspect
print("Value range:", X_train.min(), "to", X_train.max())
print(pd.Series(y_train).value_counts())

Value range: 0.0 to 1.0
0    72471
4     6431
2     5788
1     2223
3      641
Name: count, dtype: int64


In [5]:
# Reshape for 1D CNN
X_train = X_train.reshape(-1, 187, 1)
X_test = X_test.reshape(-1, 187, 1)

In [6]:
# Train/Validation Split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)
print("Train:", X_tr.shape, "Val:", X_val.shape)

Train: (70043, 187, 1) Val: (17511, 187, 1)


In [7]:
# Class weights
classes = np.unique(y_tr)
weights = compute_class_weight("balanced", classes=classes, y=y_tr)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
print(class_weights)

{0: 0.2416234023837039, 1: 7.878852643419573, 2: 3.0256155507559397, 3: 27.307212475633527, 4: 2.722759961127308}


In [8]:
# PTBDB
ptb_norm = pd.read_csv("heartbeat/ptbdb_normal.csv", header=None)
ptb_abn = pd.read_csv("heartbeat/ptbdb_abnormal.csv", header=None)

ptb = pd.concat([ptb_norm, ptb_abn], ignore_index=True)
X_ptb = ptb.iloc[:, :-1].values
y_ptb = ptb.iloc[:, -1].values.astype(int)

print(X_ptb.shape, pd.Series(y_ptb).value_counts())

(14552, 187) 1    10506
0     4046
Name: count, dtype: int64


In [9]:
# PTBDB preprocessing
X_ptb_train, X_ptb_test, y_ptb_train, y_ptb_test = train_test_split(
    X_ptb, y_ptb,
    test_size=0.2,
    stratify=y_ptb,
    random_state=42
)

X_ptb_train, X_ptb_val, y_ptb_train, y_ptb_val = train_test_split(
    X_ptb_train, y_ptb_train,
    test_size=0.2,
    stratify=y_ptb_train,
    random_state=42
)

X_ptb_train = X_ptb_train.reshape(-1, 187, 1)
X_ptb_val = X_ptb_val.reshape(-1, 187, 1)
X_ptb_test = X_ptb_test.reshape(-1, 187, 1)

print("PTBDB Train:", X_ptb_train.shape)
print("PTBDB Val:", X_ptb_val.shape)
print("PTBDB Test:", X_ptb_test.shape)

PTBDB Train: (9312, 187, 1)
PTBDB Val: (2329, 187, 1)
PTBDB Test: (2911, 187, 1)


In [10]:
# PTBDB class weights
ptb_classes = np.unique(y_ptb_train)

ptb_weights = compute_class_weight(
    class_weight="balanced",
    classes=ptb_classes,
    y=y_ptb_train
)

ptb_class_weights = {int(c): float(w) for c, w in zip(ptb_classes, ptb_weights)}

print("PTBDB Class Weights:")
print(ptb_class_weights)

PTBDB Class Weights:
{0: 1.79837775202781, 1: 0.6925479696564034}


# IoMT Pipeline

In [11]:
# Load and Inspect
df = pd.read_excel("patients_data_with_alerts.xlsx", engine="openpyxl")

print(df.shape)
print(df.head())
df.info()
print(df.isna().sum())
print(df.describe())

(50000, 13)
   Patient Number  Heart Rate (bpm)  SpO2 Level (%)  \
0               1                96              92   
1               2                76              83   
2               3                92              88   
3               4               137              84   
4               5                76              81   

   Systolic Blood Pressure (mmHg)  Diastolic Blood Pressure (mmHg)  \
0                             101                               89   
1                             103                               85   
2                             123                               70   
3                             167                               97   
4                             175                               80   

   Body Temperature (°C) Fall Detection  Predicted Disease  Data Accuracy (%)  \
0              36.904680             No  Diabetes Mellitus                 95   
1              37.254129            Yes             Asthma                

In [12]:
# Clean IoMT dataset and create triage target

# Create triage label from patient vital signs
def assign_triage(row):
    heart_rate = row["Heart Rate (bpm)"]
    spo2 = row["SpO2 Level (%)"]
    systolic = row["Systolic Blood Pressure (mmHg)"]
    diastolic = row["Diastolic Blood Pressure (mmHg)"]
    temperature = row["Body Temperature (°C)"]
    fall = str(row["Fall Detection"]).strip().lower() == "yes"

    # Highest risk
    if (
        spo2 < 90 or
        heart_rate < 40 or heart_rate >= 130 or
        systolic < 90 or systolic >= 180 or
        diastolic >= 120 or
        temperature < 35 or temperature >= 39.5
    ):
        return "Emergency"

    # Medium risk
    elif (
        spo2 < 94 or
        heart_rate < 50 or heart_rate >= 110 or
        systolic < 100 or systolic >= 140 or
        diastolic >= 90 or
        temperature < 36 or temperature >= 38 or
        fall
    ):
        return "Urgent"

    # Low risk
    else:
        return "Non-Urgent"


df["Triage"] = df.apply(assign_triage, axis=1)

target = "Triage"

# Drop columns that should not be used as model input
drop_cols = [
    "Patient Number",
    "Predicted Disease",
    "Data Accuracy (%)",
    "Heart Rate Alert",
    "SpO2 Level Alert",
    "Blood Pressure Alert",
    "Temperature Alert"
]

df = df.drop(columns=drop_cols, errors="ignore")

# Remove duplicate rows
df = df.drop_duplicates()

print("Shape after creating triage target and cleaning:", df.shape)
print("Triage distribution:")
print(df["Triage"].value_counts())

df.head()

Shape after creating triage target and cleaning: (50000, 7)
Triage distribution:
Triage
Emergency     30510
Urgent        17970
Non-Urgent     1520
Name: count, dtype: int64


,Heart Rate (bpm),SpO2 Level (%),Systolic Blood Pressure (mmHg),Diastolic Blood Pressure (mmHg),Body Temperature (°C),Fall Detection,Triage
0,96,92,101,89,36.904680,No,Urgent
1,76,83,103,85,37.254129,Yes,Emergency
2,92,88,123,70,36.539418,Yes,Emergency
3,137,84,167,97,37.018767,Yes,Emergency
4,76,81,175,80,37.328671,No,Emergency


In [13]:
# Split features and target
X = df.drop(columns=[target])
y = df[target]

print("Features:", list(X.columns))
print("Target classes:", y.unique())

# Fill missing values
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
for c in cat_cols:
    X[c] = X[c].fillna(X[c].mode()[0])

Features: ['Heart Rate (bpm)', 'SpO2 Level (%)', 'Systolic Blood Pressure (mmHg)', 'Diastolic Blood Pressure (mmHg)', 'Body Temperature (°C)', 'Fall Detection']
Target classes: ['Urgent' 'Emergency' 'Non-Urgent']


In [14]:
# Encode
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Encode target labels
label_encoder = LabelEncoder()

if y.dtype == "object":
    y = label_encoder.fit_transform(y)

label_mapping = {
    str(label): int(encoded)
    for label, encoded in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))
}
print("Target label mapping:")
print(label_mapping)
print("Shape after encoding:", X.shape)

Target label mapping:
{'Emergency': 0, 'Non-Urgent': 1, 'Urgent': 2}
Shape after encoding: (50000, 6)


In [15]:
# Split, then scale
Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
Xt_tr = scaler.fit_transform(Xt_tr)
Xt_te = scaler.transform(Xt_te)

In [16]:
# Check balance, apply SMOTE only if needed
class_counts = pd.Series(yt_tr).value_counts()
print("Before SMOTE:")
print(class_counts)

imbalance_ratio = class_counts.max() / class_counts.min()
print("Imbalance ratio:", imbalance_ratio)

if imbalance_ratio > 1.5:
    Xt_tr, yt_tr = SMOTE(random_state=42).fit_resample(Xt_tr, yt_tr)
    print("SMOTE applied.")
else:
    print("SMOTE skipped because the dataset is already balanced.")

print("Final class distribution:")
print(pd.Series(yt_tr).value_counts())

Before SMOTE:
0    24408
2    14376
1     1216
Name: count, dtype: int64
Imbalance ratio: 20.07236842105263
SMOTE applied.
Final class distribution:
2    24408
0    24408
1    24408
Name: count, dtype: int64


In [17]:
# Final Preprocessing Summary
print("========== ECG DATASET ==========")
print("MIT-BIH Train:", X_tr.shape)
print("MIT-BIH Validation:", X_val.shape)
print("MIT-BIH Test:", X_test.shape)
print("MIT-BIH Class Weights:", class_weights)

print("\nPTBDB Train:", X_ptb_train.shape)
print("PTBDB Validation:", X_ptb_val.shape)
print("PTBDB Test:", X_ptb_test.shape)
print("PTBDB Class Weights:", ptb_class_weights)

print("\n========== IoMT DATASET ==========")
print("IoMT Train:", Xt_tr.shape)
print("IoMT Test:", Xt_te.shape)
print("IoMT Target: Triage")
print("IoMT Classes:", np.unique(yt_tr))
print("Triage Label Mapping:", label_mapping)

print("\nPreprocessing steps used:")
print("1. Duplicate removal")
print("2. Missing value checking / handling")
print("3. Rule-based triage target generation")
print("4. Categorical encoding")
print("5. Target label encoding")
print("6. Feature scaling using StandardScaler")
print("7. ECG reshaping for 1D CNN")
print("8. Class imbalance checking / optional SMOTE")

========== ECG DATASET ==========
MIT-BIH Train: (70043, 187, 1)
MIT-BIH Validation: (17511, 187, 1)
MIT-BIH Test: (21892, 187, 1)
MIT-BIH Class Weights: {0: 0.2416234023837039, 1: 7.878852643419573, 2: 3.0256155507559397, 3: 27.307212475633527, 4: 2.722759961127308}

PTBDB Train: (9312, 187, 1)
PTBDB Validation: (2329, 187, 1)
PTBDB Test: (2911, 187, 1)
PTBDB Class Weights: {0: 1.79837775202781, 1: 0.6925479696564034}

========== IoMT DATASET ==========
IoMT Train: (73224, 6)
IoMT Test: (10000, 6)
IoMT Target: Triage
IoMT Classes: [0 1 2]
Triage Label Mapping: {'Emergency': 0, 'Non-Urgent': 1, 'Urgent': 2}

Preprocessing steps used:
1. Duplicate removal
2. Missing value checking / handling
3. Rule-based triage target generation
4. Categorical encoding
5. Target label encoding
6. Feature scaling using StandardScaler
7. ECG reshaping for 1D CNN
8. Class imbalance checking / optional SMOTE
